# E2.5 · Privacy and data protection

**Function E — AI Governance for Agentic Systems → Building the Governance Platform — Regulatory and Compliance**  ·  *Security of AI*

Builds on **[E2.4 · Sector overlays](https://spbreed.github.io/cyber-commons/lessons/E2.4.html)**.

| | |
|---|---|
| Tools used | Presidio, GLiNER-PII |

## What this lesson is

**What it covers.** Run PII redaction inside the trust boundary with Presidio before anything crosses out.

**Why a security engineer needs it.** Deletion when the data is in weights, not a database. The control it builds is: lawful basis, ADM rights, residency in inference and retrieval paths, retention of traces.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Prompts, context, logs and training runs are all places personal data ends up, and none of them looks like a database to the people who designed the privacy programme. Lawful basis, minimisation and retention apply to all four.

> **At CyberTravels.** Passport numbers reach CyberTravels' prompts, its context window, its vector store and its logs. None of those looks like a database to the privacy programme. R10, R12.

## 2 · The framework

```
   where personal data actually ends up

   prompt --> context window --> model --> output
      |            |               |         |
      +------------+---------------+---------+
                        |
                    the LOGS

   none of these look like a database to the privacy programme
   lawful basis . minimisation . retention . cross-border, for all four
```

Privacy for agents turns on one fact that surprises most teams: **the context
window is a disclosure, and the trace is a record.**

When an agent reads a customer record to do its job, that record enters the
model's context. If the trace is retained — and it usually is, for forensics
(D1.5) — then personal data now exists in a system that was never in the privacy
review, with a retention period nobody set, in a place the erasure process does
not reach.

Three obligations attach, and the third is the one that bites:

- **Lawful basis** for the processing that put it there.
- **Retention limit** on the trace itself, separately from the source system.
- **Erasure** — and this reaches into traces, eval corpora, fine-tuning sets and
  backups.

The capability that makes erasure possible is the same one C2.4 built for poison
removal: per-record hashes. Without them you cannot locate the record, so you
cannot delete it.

## 3 · The procedure, as a skill

Five items of personal data are in the agent trace and nobody put them there deliberately. The skill maps every system holding a copy and runs a real erasure request through all of them — three of which cannot delete one subject's records.

### The skill — [`skills/regulatory/trace-personal-data-audit/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/regulatory/trace-personal-data-audit/SKILL.md)

```yaml
name: trace-personal-data-audit
description: >-
  Find personal data that nobody deliberately placed in an agent trace, and run
  an erasure request through every system that holds a copy. Use when agent
  telemetry meets data protection, or before a subject access request arrives.
allowed-tools: Read, Grep, Glob
```

# Nobody put it there, and it is there

An agent trace accumulates personal data as a side effect: a name in a ticket, an
email in a tool result, an account number in a document it read, a card number
in a file it was asked to fix. None of it was placed deliberately, all of it is
personal data, and it is copied into every system the trace is shipped to.

## When to use this

Before agent telemetry is retained or exported, and before the first erasure
request rather than during it.

## Procedure

**1 — Run detectors over the whole trace.** Every field, every step. Names,
emails, account and card numbers, health terms. Record which step introduced
each, because that tells you whether it is preventable.

**2 — Map every system that holds a copy.** The trace store, the SIEM, the
warehouse, backups, and any vendor it is exported to. This list is the erasure
surface and it is longer than the trace store.

**3 — Run a real erasure request end to end.** Locate every copy for one
subject, delete, and verify. Systems that cannot delete a single subject's
records — append-only stores, immutable backups, aggregate indexes — are the
finding.

**4 — Say what is legitimately retained and why.** Some copies survive erasure
lawfully. Naming the basis, per system, is the difference between a defensible
position and a gap.

**5 — Reduce at source.** Redaction at write time is cheaper than erasure across
six systems. Say which detector should run before the trace is stored.

## Output contract

```json
{
  "trace": {"steps": 0},
  "detections": [{"kind": "str", "field": "str", "step": 0, "deliberate": false}],
  "systems": [{"name": "str", "holds_copy": true, "can_delete_subject": false}],
  "erasure": {"subject": "str", "located": 0, "deleted": 0, "failed_in": ["str"]},
  "retained_lawfully": [{"system": "str", "basis": "str"}],
  "redaction_at_source": ["str"]
}
```

## Failure modes

- **Auditing the trace store only.** The copies are the problem.
- **Assuming erasure works.** Run one and find out.
- **Reporting failures without the lawful retentions.** Half the list is fine
  and the report loses credibility without saying so.

In [ ]:
# The code is not in this notebook. It is the file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/regulatory/trace-personal-data-audit/scripts/trace_personal_data_audit.py
SCRIPT = "skills/regulatory/trace-personal-data-audit/scripts/trace_personal_data_audit.py"

import glob, os, subprocess, sys

# The skills tree: the attached dataset on Kaggle, the checkout locally.
_ROOTS = sorted(glob.glob("/kaggle/input/**/cyber-commons-skills", recursive=True)) + [".", "..", "../.."]
_root = next((r for r in _ROOTS if os.path.isfile(os.path.join(r, SCRIPT))), None)
if _root is None:
    raise SystemExit("skills tree not found. On Kaggle add the dataset "
                     "cybercommons/cyber-commons-skills; locally run from a checkout.")

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

Five items of personal data appear in the agent trace — name, email, account number and payment card — none placed there deliberately. The erasure request fails in three systems that cannot locate the record, two of which retain indefinitely. Building a subject index locates the affected steps, erasure leaves no personal data while retaining the hash, and per-field retention drops the sensitive field at 7 days.

## Your turn

Time-box this to an hour: can you delete one customer's data from your agent traces today? The answer usually arrives in ten minutes and is usually no — and the eval corpus is the system people forget entirely.

---

**Next → [E2.6 · Incident and disclosure obligations](https://spbreed.github.io/cyber-commons/lessons/E2.6.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/E2.5.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/E2.5.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*